In [1]:
import pathlib
import logging
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
# Register the parent directory as a path to look for modules
notebook_dir = pathlib.Path().resolve()
parent_dir = notebook_dir.parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from app.utils.dataset_builder.landsat_dataset_builder import LandsatDataBuilder
from app.models.file_processing.sources import FileSourceConfig
from app.utils.image_transformation.image_cube_operations import ImageCubeOperations, CubeRepresentation

logger = logging.getLogger("notebook")
logger.setLevel(logging.INFO)
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)
import boto3

In [2]:
from app.utils.patch_generation.final.final_patcher import FinalPatchShuffler

finals = FinalPatchShuffler(
    intermediate_patch_s3_key="patches/intermediate/s200w128h128s64/",
    final_s3_key="/asdsa/aasd/"
)

/Users/ashwinravi/Desktop/Code Repos/hsi-anomaly-foundations-allotrope/.venv/lib/python3.14/site-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


In [3]:
sample = next(iter(finals.dataset))

download failed: s3://allotrope-raw-data-india/patches/intermediate/s200w128h128s64/intermediate_shard_0134.tar to - [Errno 32] Broken pipe


In [ ]:

import webdataset as wds

# 1. Point directly to the exact S3 URL that threw the error (no brace expansion)
url = "pipe:aws s3 cp s3://allotrope-raw-data-india/patches/intermediate/s200w128h128s64/intermediate_shard_0190.tar -"

# 2. Create a barebones dataset: no workers, no resampling, no shuffling
dataset = wds.WebDataset(url).decode()

print("Opening pipe to intermediate_shard_0190.tar...\n")
try:
    # 3. Iterate through the patches sequentially
    for i, sample in enumerate(dataset):
        
        # Print a success message for the first few patches to verify the stream is working
        if i < 3:
            print(f"Successfully read patch {i}!")
            print(f"  Keys found: {list(sample.keys())}")
            if '.npy' in sample:
                print(f"  Array shape: {sample['.npy'].shape}")
                
        # If we successfully read 100 patches, the file is likely perfectly healthy
        if i == 100:
            print("\nSuccessfully read 100 patches in a row. The file is intact.")
            print("The Broken Pipe error was likely caused by an Out-Of-Memory (OOM) crash.")
            break

except Exception as e:
    # If the file is corrupted, this will catch the exact moment it fails
    print(f"\n🚨 CRASH DETECTED at patch {i}:")
    print(f"Error details: {e}")
    print("Conclusion: The TAR file or one of its contents is corrupted.")

In [5]:
s3_client = boto3.client("s3", region_name="ap-south-1")
paginator = s3_client.get_paginator("list_objects_v2")

In [6]:
page_iterator = paginator.paginate(

    Bucket="allotrope-raw-data-india",
    Prefix = "patches/intermediate/s200w128h128s64/",
    PaginationConfig = {
        "PageSize": 500
    }
)

In [8]:
from pprint import pprint
for page in page_iterator:
    pprint(page)
    break
    

{'Contents': [{'ChecksumAlgorithm': ['CRC32'],
               'ChecksumType': 'COMPOSITE',
               'ETag': '"e06ef4b36073598af1ce64b2ddd57522-138"',
               'Key': 'patches/intermediate/s200w128h128s64/intermediate_shard_0000.tar',
               'LastModified': datetime.datetime(2026, 2, 27, 9, 3, 29, tzinfo=tzutc()),
               'Size': 1150556160,
               'StorageClass': 'STANDARD'},
              {'ChecksumAlgorithm': ['CRC32'],
               'ChecksumType': 'COMPOSITE',
               'ETag': '"bd54c78374664d2e0e5bebdeba0c72d7-138"',
               'Key': 'patches/intermediate/s200w128h128s64/intermediate_shard_0001.tar',
               'LastModified': datetime.datetime(2026, 2, 27, 9, 4, 1, tzinfo=tzutc()),
               'Size': 1150556160,
               'StorageClass': 'STANDARD'},
              {'ChecksumAlgorithm': ['CRC32'],
               'ChecksumType': 'COMPOSITE',
               'ETag': '"6622b91b6b2f6b1e52eac7f55b9e6efe-138"',
               'K

In [ ]:
from app.utils.patch_generation.intermediate.landsat_intermediate_patcher import LandsatIntermediateSharder

In [ ]:
patcher = LandsatIntermediateSharder(
    source_folder = "/Users/ashwinravi/Desktop/",
    destination_folder = "/Users/ashwinravi/Desktop/",
    destination_prefix="patches/intermediate/test1"

)
patcher.sharder(scenes = 1)

# writing /Users/ashwinravi/Desktop/intermediate_shard_0000.tar 0 0.0 GB 0


Scene Number:   0%|          | 0/1 [00:00<?, ?it/s]

['landsat/LC91440372024030LGN00/LC09_L2SP_144037_20240130_20240131_02_T1_QA_PIXEL.TIF', 'landsat/LC91440372024030LGN00/LC09_L2SP_144037_20240130_20240131_02_T1_ST_B10.TIF']
Using device: mps
Max Temp = 26.27705955505371
Min Temp = -124.1500015258789
Masked Array
(40106003, 1)
[-26.02206421 -15.44670963   0.45391911   6.74307585   9.68257332]
Anchors Set to [[-26.02206421]
 [-15.44670963]
 [  0.45391911]
 [  6.74307585]
 [  9.68257332]]
-11.546080887317657
[0]


Uploading /Users/ashwinravi/Desktop/intermediate_shard_0000.tar to s3://allotrope-raw-data-india/patches/intermediate/test1/intermediate_shard_0000.tar...


In [ ]:
next(patches)

{'__key__': 'LC09_L2SP_130053_20240417_20240418_02_T1_ST_B10#row_coord:192#col_coord:1536',
 'meta.json': {'scene_id': 'LC09_L2SP_130053_20240417_20240418_02_T1_ST_B10',
  'row_coords': 192,
  'col_coords': 1536,
  'patch_height': 128,
  'patch_width': 128,
  'patch_stride': 64,
  'bands': 1},
 'pixels.npy': array([[[30.785429, 30.775175, 30.76834 , ..., 30.836699, 30.864042,
          30.97342 ],
         [30.819609, 30.81619 , 30.809355, ..., 30.823027, 30.86746 ,
          30.987091],
         [30.864042, 30.864042, 30.860624, ..., 30.823027, 30.877716,
          30.987091],
         ...,
         [30.594019, 30.583765, 30.635036, ..., 30.788847, 30.75125 ,
          30.723904],
         [30.532495, 30.539331, 30.594019, ..., 30.81619 , 30.7991  ,
          30.785429],
         [30.511988, 30.529078, 30.57693 , ..., 30.840117, 30.840117,
          30.829863]]], shape=(1, 128, 128), dtype=float32),
 'validity_cube.npy': array([[[1, 1, 1, ..., 1, 1, 1],
         [1, 1, 1, ..., 1, 1, 1

{'qa_pixel': '/Users/ashwinravi/Desktop/LC09_L2SP_130053_20240417_20240418_02_T1_QA_PIXEL.TIF',
 'b10': '/Users/ashwinravi/Desktop/LC09_L2SP_130053_20240417_20240418_02_T1_ST_B10.TIF'}